In [1]:
import torch
from PIL import Image
import os
from torchvision.transforms.v2 import PILToTensor

os.chdir("..") # to allow relative imports
from model.swinv2_encoder import *

torch.set_printoptions(linewidth=10000000, threshold=300000000)

/home/charles/.local/share/envs/sarformer/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/charles/.local/share/envs/sarformer/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: DeprecationWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", DeprecationWarning)


In [2]:
x = torch.arange(36).reshape(3, 3, 4) # B Ph*Pw C
x[0, 1, :] = -1
x[2, 2, :] = -1
x[0, 0, :] = 0
x[2, 0, :] = 0
x[2, 2, :] = 0
x

tensor([[[ 0,  0,  0,  0],
         [-1, -1, -1, -1],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]],

        [[ 0,  0,  0,  0],
         [28, 29, 30, 31],
         [ 0,  0,  0,  0]]])

In [3]:
sorted, abs_sorted_idxs = torch.sort(torch.abs(x), dim=1)
print(sorted, end="\n")
abs_sorted_idxs 

tensor([[[ 0,  0,  0,  0],
         [ 1,  1,  1,  1],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]],

        [[ 0,  0,  0,  0],
         [ 0,  0,  0,  0],
         [28, 29, 30, 31]]])


tensor([[[0, 0, 0, 0],
         [1, 1, 1, 1],
         [2, 2, 2, 2]],

        [[0, 0, 0, 0],
         [1, 1, 1, 1],
         [2, 2, 2, 2]],

        [[0, 0, 0, 0],
         [2, 2, 2, 2],
         [1, 1, 1, 1]]])

In [8]:
t = torch.gather(x, 1, abs_sorted_idxs)[:, 2:]
(t == 0).all(dim=2).count_nonzero()

tensor(0)

In [52]:
from time import time
B = 3
num_partial_masked = 2
num_patch_specks = 3

times = []
for _ in range(1000):
    start = time()
    partial_patch_batch_idxs = torch.arange(B).repeat_interleave(
        num_partial_masked * num_patch_specks
    )
    end = time()
    times.append(end - start)

sum(times) / len(times)

2.0128250122070314e-05

In [53]:
times = []
for _ in range(1000):
    start = time()
    partial_patch_batch_idxs = torch.arange(B).view(1, B).expand(num_partial_masked * num_patch_specks, B).contiguous().view(1, -1)
    partial_patch_batch_idxs
    end = time()
    times.append(end - start)

sum(times) / len(times)

2.4998188018798827e-05

In [11]:
x = torch.arange(27).reshape(3, 3, 3)
x = F.pad(x, (0, 0, 0, -2, 0, 0))
x.shape

torch.Size([3, 1, 3])

In [3]:
from PIL import Image
from torch import int8
from torchvision.transforms.functional import pil_to_tensor, to_pil_image

im = Image.open("notebooks/NAIP_23454.jpg")
x = pil_to_tensor(im).unsqueeze(0) #.to(torch.float32)

swin = SwinTransformerV2(
    img_size=x.shape[-1],
    patch_size=16,
    in_chans=3,
    depths=[2, 2, 18, 2],
    window_size=4,
    pretrained_window_sizes=[0, 0, 0, 0],
    ape=True,
    mask_token=0,
    mask_proportion=0.,
)
masked_full_x, _, _ = swin.pre_patch_embed_mask(x)
masked_full_x = masked_full_x.squeeze(0).to(torch.uint8)
to_pil_image(masked_full_x)

AssertionError: resolution 24 has to be a muliple of 8 and the window size at the same time